# ActiveMQ Artemis Shared-Store HA (Active/Passive) with Docker Compose

This notebook provides a copy/paste-friendly guide to run **2 ActiveMQ Artemis brokers** in **active/passive** mode using **shared storage** ("shared-store" HA).

## What you get
- **1 LIVE (active)** broker
- **1 BACKUP (passive)** broker
- Both mount the **same journal directory** (shared volume)
- Automatic failover when the live broker stops

## When to use
- Simple lab setups
- Single-host deployments
- Shared storage available (NFS/SAN/…)

> For Kubernetes/cloud, **replication HA** is often a better fit.


## 1) Project layout

Create a folder and put a `docker-compose.yml` inside:

```
artemis-sharedstore-ha/
  docker-compose.yml
```


## 2) `docker-compose.yml`

Save the following as `docker-compose.yml`.

**Important:** both services mount the **same named volume** `artemis-data` to the Artemis instance directory. Only one broker can hold the journal lock at a time.


In [ ]:
docker_compose_yml = r'''version: "3.8"

services:
  artemis-live:
    image: apache/activemq-artemis:latest
    container_name: artemis-live
    hostname: artemis-live
    environment:
      ARTEMIS_USER: admin
      ARTEMIS_PASSWORD: admin
      ANONYMOUS_LOGIN: "false"
      EXTRA_ARGS: >
        --host artemis-live
        --clustered
        --shared-store
        --failover-on-shutdown
    volumes:
      - artemis-data:/var/lib/artemis-instance
    ports:
      - "61616:61616"   # JMS Core
      - "8161:8161"     # Web console
    networks:
      - artemis-net

  artemis-backup:
    image: apache/activemq-artemis:latest
    container_name: artemis-backup
    hostname: artemis-backup
    depends_on:
      - artemis-live
    environment:
      ARTEMIS_USER: admin
      ARTEMIS_PASSWORD: admin
      ANONYMOUS_LOGIN: "false"
      EXTRA_ARGS: >
        --host artemis-backup
        --clustered
        --shared-store
        --slave
    volumes:
      - artemis-data:/var/lib/artemis-instance
    networks:
      - artemis-net

volumes:
  artemis-data:

networks:
  artemis-net:
'''
print(docker_compose_yml)


## 3) Start the brokers

From inside `artemis-sharedstore-ha/`:

```bash
docker compose up -d
docker logs -f artemis-live
docker logs -f artemis-backup
```

Expected behavior:
- `artemis-live` starts and becomes **active**
- `artemis-backup` remains **passive** (waiting)


## 4) Test failover

Stop the live broker:

```bash
docker stop artemis-live
docker logs -f artemis-backup
```

Expected:
- backup acquires the journal lock and becomes **active**

> Tip: If you kill the container abruptly, failover can still work, but **`--failover-on-shutdown`** helps clean handoff on normal stops.


## 5) Client failover URL (IMPORTANT)

To reconnect automatically after failover, clients should use a **failover URL**.

### Inside the Compose network
```
failover:(tcp://artemis-live:61616,tcp://artemis-backup:61616)
```

### From the host machine
In this simple compose file, only the live broker is published on the host (`61616`).

If you also want to reach the backup directly from the host, add a port mapping for it, e.g.:

```yaml
ports:
  - "61617:61616"
```

Then host-side failover would be:
```
failover:(tcp://localhost:61616,tcp://localhost:61617)
```


## 6) Notes & gotchas

- Shared-store HA requires **real shared storage** in production (NFS/SAN/cluster FS). A local Docker named volume is fine for demos on one host.
- Only one broker can be active because only one can hold the **journal lock**.
- For orchestration (Kubernetes), shared-store is usually harder than **replication HA**.
